# Phase 4 — Proper PyTorch Engineering

Now we do things the *right* way:
- `nn.Module` model
- `Dataset` + `DataLoader`
- clean train/eval loops
- metrics + logging
- reproducibility (seeds, determinism)
- initialization
- gradient clipping
- LR scheduler
- checkpoint save/load
- (optional) AMP mixed precision if CUDA

We'll still use XOR (tiny) to keep focus on engineering patterns, then show how to scale the same skeleton.


## 0) Setup

In [ ]:
import os
import random
import math
from dataclasses import dataclass
from typing import Dict, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt

torch.set_printoptions(precision=4, sci_mode=False)
np.set_printoptions(precision=4, suppress=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


## 1) Reproducibility utilities

Determinism matters when you debug. For GPU, full determinism can reduce performance.


In [ ]:
def seed_everything(seed: int = 0, deterministic: bool = True):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        torch.use_deterministic_algorithms(True, warn_only=True)
    else:
        torch.backends.cudnn.benchmark = True

seed_everything(42, deterministic=True)


## 2) Config (single place for hyperparameters)

In [ ]:
@dataclass
class Config:
    seed: int = 42
    batch_size: int = 4
    hidden: int = 8
    lr: float = 0.5
    weight_decay: float = 0.0
    epochs: int = 4000
    grad_clip_norm: float = 1.0  # set 0 to disable
    scheduler_step: int = 1000
    scheduler_gamma: float = 0.5
    use_amp: bool = True  # only effective on CUDA

cfg = Config()
cfg


## 3) Dataset + DataLoader

Even for XOR, we use the proper pattern.


In [ ]:
class XORDataset(Dataset):
    def __init__(self):
        self.X = torch.tensor([[0.,0.],[0.,1.],[1.,0.],[1.,1.]], dtype=torch.float32)
        self.y = torch.tensor([[0.],[1.],[1.],[0.]], dtype=torch.float32)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

ds = XORDataset()
dl = DataLoader(ds, batch_size=cfg.batch_size, shuffle=True, drop_last=False)

xb, yb = next(iter(dl))
print("batch X:", xb.shape)
print("batch y:", yb.shape)


## 4) Model as `nn.Module`

We'll build a tiny MLP. Use logits (no sigmoid inside) + `BCEWithLogitsLoss` for stability.


In [ ]:
class MLPBinary(nn.Module):
    def __init__(self, in_dim: int, hidden: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.Tanh(),               # try ReLU too
            nn.Linear(hidden, 1)     # logits
        )

    def forward(self, x):
        return self.net(x)

model = MLPBinary(in_dim=2, hidden=cfg.hidden).to(device)
model


### 4.1 Initialization (explicit)

Good init avoids saturation. We'll use Xavier for tanh.


In [ ]:
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)  # good default for tanh
        nn.init.zeros_(m.bias)

model.apply(init_weights)


## 5) Loss, optimizer, scheduler

In [ ]:
criterion = nn.BCEWithLogitsLoss()  # expects logits, applies sigmoid internally safely

optimizer = optim.SGD(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay, momentum=0.0)

scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=cfg.scheduler_step, gamma=cfg.scheduler_gamma)

# AMP (automatic mixed precision) is only meaningful on CUDA
use_amp = cfg.use_amp and device.type == "cuda"
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

use_amp, device


## 6) Metrics

Accuracy for XOR is fine. For real problems, add F1/AUROC etc.


In [ ]:
@torch.no_grad()
def accuracy_from_logits(logits: torch.Tensor, y: torch.Tensor) -> float:
    probs = torch.sigmoid(logits)
    preds = (probs > 0.5).to(y.dtype)
    return (preds == y).float().mean().item()


## 7) Train / Eval loops

This is the reusable skeleton:
- `model.train()` vs `model.eval()`
- zero_grad
- forward
- loss
- backward
- optional grad clip
- optimizer step
- scheduler step (per-epoch here)
- log metrics


In [ ]:
def train_one_epoch(model, dl, optimizer, criterion, scaler, grad_clip_norm: float = 0.0):
    model.train()
    total_loss = 0.0
    total_acc = 0.0
    n = 0

    for xb, yb in dl:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=scaler.is_enabled()):
            logits = model(xb)
            loss = criterion(logits, yb)

        scaler.scale(loss).backward()

        if grad_clip_norm and grad_clip_norm > 0:
            scaler.unscale_(optimizer)  # required before clipping with AMP
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip_norm)

        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * xb.size(0)
        total_acc += accuracy_from_logits(logits, yb) * xb.size(0)
        n += xb.size(0)

    return {"loss": total_loss / n, "acc": total_acc / n}

@torch.no_grad()
def eval_one_epoch(model, dl, criterion):
    model.eval()
    total_loss = 0.0
    total_acc = 0.0
    n = 0

    for xb, yb in dl:
        xb = xb.to(device)
        yb = yb.to(device)

        logits = model(xb)
        loss = criterion(logits, yb)

        total_loss += loss.item() * xb.size(0)
        total_acc += accuracy_from_logits(logits, yb) * xb.size(0)
        n += xb.size(0)

    return {"loss": total_loss / n, "acc": total_acc / n}


## 8) Fit loop + plots

In [ ]:
seed_everything(cfg.seed, deterministic=True)

model = MLPBinary(in_dim=2, hidden=cfg.hidden).to(device)
model.apply(init_weights)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.SGD(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=cfg.scheduler_step, gamma=cfg.scheduler_gamma)

use_amp = cfg.use_amp and device.type == "cuda"
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

history = {"train_loss": [], "train_acc": [], "lr": []}

for epoch in range(1, cfg.epochs + 1):
    tr = train_one_epoch(model, dl, optimizer, criterion, scaler, grad_clip_norm=cfg.grad_clip_norm)
    scheduler.step()

    history["train_loss"].append(tr["loss"])
    history["train_acc"].append(tr["acc"])
    history["lr"].append(scheduler.get_last_lr()[0])

    if epoch % 500 == 0 or epoch == 1 or epoch == cfg.epochs:
        print(f"epoch={epoch:4d}  loss={tr['loss']:.6f}  acc={tr['acc']:.2f}  lr={history['lr'][-1]:.5f}")

plt.figure()
plt.plot(history["train_loss"])
plt.title("Training loss")
plt.xlabel("epoch")
plt.ylabel("BCEWithLogitsLoss")
plt.show()

plt.figure()
plt.plot(history["train_acc"])
plt.title("Training accuracy")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.show()

plt.figure()
plt.plot(history["lr"])
plt.title("Learning rate schedule")
plt.xlabel("epoch")
plt.ylabel("lr")
plt.show()


## 9) Inference: inspect predictions

In [ ]:
@torch.no_grad()
def predict_probs(model, X):
    model.eval()
    X = X.to(device)
    logits = model(X)
    return torch.sigmoid(logits)

X_all = torch.tensor([[0.,0.],[0.,1.],[1.,0.],[1.,1.]], dtype=torch.float32)
probs = predict_probs(model, X_all).cpu()
print("X:\n", X_all)
print("probs:", probs.T)
print("preds:", (probs > 0.5).int().T)


## 10) Checkpointing: save/load

For real work, always save:
- model state
- optimizer state
- scheduler state
- config
- epoch


In [ ]:
ckpt_path = "/mnt/data/phase4_xor_checkpoint.pt"

def save_checkpoint(path, model, optimizer, scheduler, cfg, epoch):
    payload = {
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "cfg": cfg.__dict__,
        "epoch": epoch,
    }
    torch.save(payload, path)

def load_checkpoint(path, model, optimizer=None, scheduler=None):
    payload = torch.load(path, map_location=device)
    model.load_state_dict(payload["model"])
    if optimizer is not None and "optimizer" in payload:
        optimizer.load_state_dict(payload["optimizer"])
    if scheduler is not None and "scheduler" in payload:
        scheduler.load_state_dict(payload["scheduler"])
    return payload

save_checkpoint(ckpt_path, model, optimizer, scheduler, cfg, epoch=cfg.epochs)
print("Saved:", ckpt_path, "size(bytes)=", os.path.getsize(ckpt_path))

# Demonstrate load into a fresh model
model2 = MLPBinary(in_dim=2, hidden=cfg.hidden).to(device)
model2.apply(init_weights)

opt2 = optim.SGD(model2.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
sch2 = optim.lr_scheduler.StepLR(opt2, step_size=cfg.scheduler_step, gamma=cfg.scheduler_gamma)

payload = load_checkpoint(ckpt_path, model2, opt2, sch2)
print("Loaded epoch:", payload["epoch"])
print("Loaded cfg:", payload["cfg"])

probs2 = predict_probs(model2, X_all).cpu()
print("probs2:", probs2.T)


## 11) Common failure modes (quick checklist)

- Loss not decreasing:
  - LR too high/low
  - wrong dtype (ints instead of floats)
  - forgot `optimizer.zero_grad()`
  - using sigmoid + `BCELoss` without clipping (use `BCEWithLogitsLoss`)
- Accuracy stuck:
  - model too small
  - activation saturating (bad init)
  - data/labels mismatch
- CUDA errors:
  - tensors on different devices
  - OOM: batch/model too big


## 12) Extensions (recommended)

1) Swap `Tanh` → `ReLU` and update init to Kaiming.
2) Replace XOR with a bigger synthetic dataset (e.g., two moons) and reuse the same pipeline.
3) Add validation split, early stopping, and best-checkpoint saving.
4) Add TensorBoard logging (outside notebook).
